### 02_etl_sih.ipynb — Seção 1: Setup e Funções Utilitárias

 Projeto: Análise Preditiva de Risco de Letalidade por DRSAI

 Responsabilidades desta seção:
   1. Imports organizados por origem
   2. Configuração centralizada (constantes, paths, parâmetros)
   3. Funções utilitárias validadas no spike
   4. Verificação do ambiente antes de processar

In [3]:
# Imports
#
# Organização por origem:
#   - stdlib: vem com Python, sem instalação
#   - terceiros: precisam de pip install
#   - internos: arquivos do próprio projeto

# stdlib
import os
import gc
import time
import logging
from pathlib import Path
from datetime import datetime
 
# terceiros
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa
from pysus.ftp.databases.sih import SIH

# verificação de versões mínimas
assert pd.__version__ >= "1.5", f"Pandas >= 1.5 necessário, encontrado: {pd.__version__}"
assert pa.__version__ >= "10.0", f"PyArrow >= 10.0 necessário, encontrado: {pa.__version__}"
 
print("✅ Imports OK")
print(f"   pandas  {pd.__version__}")
print(f"   pyarrow {pa.__version__}")
 

✅ Imports OK
   pandas  2.3.3
   pyarrow 23.0.1


In [4]:
# Configuração de logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    handlers=[
        logging.StreamHandler(),                    # imprime no notebook
        logging.FileHandler("pipeline_sih.log"),    # salva em arquivo
    ]
)
log = logging.getLogger(__name__)
log.info("Pipeline SIH-SUS iniciado")


17:31:19 [INFO] Pipeline SIH-SUS iniciado


In [5]:
# Constantes e configuração centralizada
# qualquer valor que possa mudar

# --- Escopo temporal ---
# Decisão: excluir 2020-2022 (choque COVID)
ANOS_BASELINE   = [2017, 2018, 2019]        # para calcular µ e σ
ANOS_MODELAGEM  = [2023, 2024]              # para aplicar o target
ANOS_TODOS      = ANOS_BASELINE + ANOS_MODELAGEM
MESES           = list(range(1, 13))
 
# --- Escopo geográfico ---
UFS_BR = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]
assert len(UFS_BR) == 27, "Deveria ter 27 UFs"
 
# Total de arquivos esperados
N_ARQUIVOS_ESPERADOS = len(ANOS_TODOS) * len(MESES) * len(UFS_BR)
assert N_ARQUIVOS_ESPERADOS == 1620, f"Esperado 1620, calculado {N_ARQUIVOS_ESPERADOS}"
 
# --- CIDs DRSAI ---
# Fonte: Funasa / Ministério da Saúde (Manual de Saneamento, 4ª ed., 2019)
# Decisão metodológica: excluir DRSAI vetorial (dengue/zika/chik) — trabalho futuro
PREFIXOS_DRSAI = {
    "fecal_oral":   ["A00", "A01", "A02", "A03", "A04", "A05",
                     "A06", "A07", "A08", "A09", "B15"],
    "leptospirose": ["A27"],
    "helmintiase":  ["B65", "B66", "B67", "B68", "B69", "B70",
                     "B71", "B72", "B73", "B74", "B75", "B76",
                     "B77", "B78", "B79", "B80", "B81", "B82", "B83"],
}
# Set para lookup O(1) — importante em loops de milhões de linhas
TODOS_PREFIXOS_DRSAI = {p for lista in PREFIXOS_DRSAI.values() for p in lista}
 
# --- Colunas selecionadas ---
# Validadas no spike: 17 colunas, 917 bytes/linha, redução 6.8x de footprint
COLUNAS_SIH = [
    "ANO_CMPT", "MES_CMPT", "DT_INTER", "DT_SAIDA",   # temporal
    "MUNIC_RES", "UF_ZI",                               # geográfico (residência)
    "DIAG_PRINC", "DIAG_SECUN", "MORTE",                # DRSAI + target
    "IDADE", "COD_IDADE", "SEXO", "RACA_COR",           # demográfico
    "DIAS_PERM", "VAL_TOT", "UTI_MES_TO",               # métricas hospitalares
    "N_AIH",                                             # auditoria
]
assert len(COLUNAS_SIH) == 17, f"Esperado 17 colunas, encontrado {len(COLUNAS_SIH)}"
 
# --- Paths ---
# Estrutura de pastas do projeto (Cookiecutter Data Science)
ROOT        = Path(".").resolve()
DATA_RAW    = ROOT / "data" / "raw" / "sih"
DATA_INTERIM = ROOT / "data" / "interim" / "sih_drsai"
DATA_PROCESSED = ROOT / "data" / "processed"
LOG_PATH    = ROOT / "pipeline_sih.log"
 
# Cria estrutura de pastas se não existir
for pasta in [DATA_RAW, DATA_INTERIM, DATA_PROCESSED]:
    pasta.mkdir(parents=True, exist_ok=True)
 
print(f"✅ Configuração OK")
print(f"   Anos baseline:  {ANOS_BASELINE}")
print(f"   Anos modelagem: {ANOS_MODELAGEM}")
print(f"   UFs: {len(UFS_BR)} | Meses: {len(MESES)} | Arquivos esperados: {N_ARQUIVOS_ESPERADOS}")
print(f"   CIDs DRSAI: {len(TODOS_PREFIXOS_DRSAI)} prefixos em {len(PREFIXOS_DRSAI)} categorias")
print(f"   Colunas SIH: {len(COLUNAS_SIH)}")

✅ Configuração OK
   Anos baseline:  [2017, 2018, 2019]
   Anos modelagem: [2023, 2024]
   UFs: 27 | Meses: 12 | Arquivos esperados: 1620
   CIDs DRSAI: 31 prefixos em 3 categorias
   Colunas SIH: 17


In [7]:
# Funções utilitárias (promovidas do spike para produção)
#
# Três funções validadas empiricamente no spike:
#   - categorizar_drsai: classificação por CID-10
#   - normalizar_tipos_sih: correção de tipos (bug do MORTE)
#   - baixar_e_ler_sih: download + column projection
#
# Uma função nova:
#   - arquivo_valido: validação de integridade (Pergunta 3 de design)
 
def categorizar_drsai(cid: str) -> str:
    """
    Classifica um CID-10 em categoria DRSAI ou 'fora_escopo'.
    
    Decisão metodológica: usa DIAG_PRINC apenas (não DIAG_SECUN).
    Justificativa: evita dupla contagem; DIAG_PRINC é a causa primária
    da internação (Manual Técnico SIH-SUS, DataSUS).
    """
    if not isinstance(cid, str) or len(cid) < 3:
        return "fora_escopo"
    prefixo = cid[:3]
    for categoria, lista in PREFIXOS_DRSAI.items():
        if prefixo in lista:
            return categoria
    return "fora_escopo"
 
 
def normalizar_tipos_sih(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza tipos das colunas SIH-SUS após leitura do parquet.
    
    Contexto: PySUS converte DBC→Parquet preservando tipos originais.
    No DBC, campos numéricos vêm como strings ('0', '1').
    Comparar string com int (df['MORTE'] == 1) retorna sempre False — 
    bug silencioso que zeraria toda a letalidade do projeto.
    
    Validado no spike: MORTE vinha como object (string), não int.
    """
    df = df.copy()
 
    # Campos binários (0/1) — NaN → 0 (conservador: assume vivo)
    if "MORTE" in df.columns:
        df["MORTE"] = (
            pd.to_numeric(df["MORTE"], errors="coerce")
              .fillna(0)
              .astype("int8")
        )
 
    # Inteiros pequenos — preserva NaN (Int8 nullable)
    for col in ["SEXO", "RACA_COR", "COD_IDADE"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int8")
 
    # Inteiros maiores
    for col in ["IDADE", "DIAS_PERM", "UTI_MES_TO", "ANO_CMPT", "MES_CMPT"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int32")
 
    # Floats
    if "VAL_TOT" in df.columns:
        df["VAL_TOT"] = pd.to_numeric(df["VAL_TOT"], errors="coerce")
 
    # Datas — formato DataSUS: YYYYMMDD como string
    for col in ["DT_INTER", "DT_SAIDA"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format="%Y%m%d", errors="coerce")
 
    # Strings
    for col in ["DIAG_PRINC", "DIAG_SECUN", "MUNIC_RES", "UF_ZI", "N_AIH"]:
        if col in df.columns:
            df[col] = df[col].astype("string")
 
    return df
 
 
def arquivo_valido(caminho: Path) -> bool:
    """
    Verifica integridade de um arquivo parquet local.
    
    Usa pq.read_schema() — lê apenas o footer (metadata), não os dados.
    Muito mais rápido que read_table() para validação em massa.
    Falha se o arquivo estiver corrompido (download interrompido, disco cheio).
    
    Decisão de design (Pergunta 3): resolvemos o Problema A (corrupção local)
    mas documentamos o Problema B (atualização retroativa DataSUS) como
    limitação — risco baixo para dados de 2017-2019, moderado para 2023-2024.
    """
    if not caminho.exists():
        return False
    try:
        pq.read_schema(caminho)
        return True
    except Exception:
        return False
 
 
def baixar_e_ler_sih(
    uf: str,
    ano: int,
    mes: int,
    pasta_destino: Path,
) -> pd.DataFrame:
    """
    Baixa um arquivo SIH-SUS (grupo RD) e retorna DataFrame normalizado.
    
    Integra: column projection (17 colunas) + normalização de tipos.
    Normalização integrada aqui garante que toda leitura retorna
    tipos corretos por padrão — sem depender da disciplina do chamador.
    
    Tenta API recente primeiro, cai para legado se necessário.
    """
    pasta_destino.mkdir(parents=True, exist_ok=True)
 
    try:
        sih = SIH().load()
        arquivos_meta = sih.get_files(group="RD", uf=uf, year=ano, month=mes)
        if not arquivos_meta:
            raise FileNotFoundError(f"Sem arquivos RD para {uf} {ano}-{mes:02d}")
        parquet_set = sih.download(arquivos_meta, local_dir=str(pasta_destino))
        if isinstance(parquet_set, list):
            parquet_set = parquet_set[0]
        df = pq.read_table(parquet_set.path, columns=COLUNAS_SIH).to_pandas()
 
    except (ImportError, AttributeError):
        # Fallback para API legada
        from pysus.online_data import SIH as SIH_legado
        sih = SIH_legado()
        parquet_set = sih.download(
            states=[uf], years=[ano], months=[mes], groups=["RD"]
        )
        if isinstance(parquet_set, list):
            parquet_set = parquet_set[0]
        df = pq.read_table(parquet_set.path, columns=COLUNAS_SIH).to_pandas()
 
    return normalizar_tipos_sih(df)
 
 
print("✅ Funções utilitárias carregadas:")
print("   - categorizar_drsai(cid)")
print("   - normalizar_tipos_sih(df)")
print("   - arquivo_valido(caminho)")
print("   - baixar_e_ler_sih(uf, ano, mes, pasta)")

✅ Funções utilitárias carregadas:
   - categorizar_drsai(cid)
   - normalizar_tipos_sih(df)
   - arquivo_valido(caminho)
   - baixar_e_ler_sih(uf, ano, mes, pasta)


In [8]:
# Smoke test das funções (roda em < 10 segundos)
# Usa o cache local do Acre/2024-06 baixado no spike.
# Se não existir, baixa de novo (arquivo pequeno, ~16KB).
 
def smoke_test_secao1() -> bool:
    """
    Smoke test das funções utilitárias da Seção 1.
    Usa Acre/jun-2024 por ser o menor arquivo (4.675 linhas).
    """
    log.info("Rodando smoke test da Seção 1...")
 
    try:
        # Verifica se já existe no cache do spike
        pasta_teste = DATA_RAW / "spike"
        df = baixar_e_ler_sih("AC", 2024, 6, pasta_teste)
 
        # Testa normalização de tipos
        assert df["MORTE"].dtype == "int8", \
            f"MORTE deveria ser int8, encontrado: {df['MORTE'].dtype}"
        assert "DIAG_PRINC" in df.columns, "DIAG_PRINC não encontrado"
        assert len(df) > 0, "DataFrame vazio"
 
        # Testa categorização
        cats = df["DIAG_PRINC"].apply(categorizar_drsai)
        assert cats.notna().all(), "categorizar_drsai retornou NaN"
 
        # Testa resultado do spike (regressão)
        n_drsai = (cats != "fora_escopo").sum()
        assert 50 <= n_drsai <= 200, \
            f"DRSAI no Acre/jun-2024 fora da faixa esperada (50-200): {n_drsai}"
 
        log.info(f"Smoke test OK: {len(df):,} linhas, {n_drsai} DRSAI")
 
        # Libera memória explicitamente
        del df, cats
        gc.collect()
 
        return True
 
    except Exception as e:
        log.error(f"Smoke test FALHOU: {e}")
        return False
 
 
smoke_ok = smoke_test_secao1()
if not smoke_ok:
    raise RuntimeError(
        "Smoke test da Seção 1 falhou. "
        "Verifique as funções utilitárias antes de prosseguir."
    )
 
print("\n" + "=" * 60)
print("✅ SEÇÃO 1 COMPLETA — todas as verificações passaram")
print("   Pode prosseguir para a Seção 2 (download em massa)")
print("=" * 60)

17:33:36 [INFO] Rodando smoke test da Seção 1...
RDAC2406.parquet: 100%|██████████| 16.4k/16.4k [00:00<00:00, 23.9kB/s]
17:33:40 [INFO] Smoke test OK: 4,675 linhas, 97 DRSAI



✅ SEÇÃO 1 COMPLETA — todas as verificações passaram
   Pode prosseguir para a Seção 2 (download em massa)


In [15]:
# CÉLULA 7 — Macrorregiões: load + validação
# Versão autossuficiente: inclui o dicionário de rename e a função.

from pathlib import Path

# Ajuste este caminho para onde o arquivo está no seu projeto
CAMINHO_MACRO = Path("/home/victoria/Documentos/Tópicos de banco de dados/projeto-datasus-drsai/data/raw/ibge/macrorregioes_saude_ms_seidigi_2024.csv")

# Mapeamento: nomes originais → snake_case
COLUNAS_MACRO_RENAME = {
    "Codigo Municipio":             "cod_municipio",
    "Municipio":                    "nome_municipio",
    "Codigo Macrorregiao de Saude": "cod_macrorregiao",
    "Macrorregiao de Saude":        "nome_macrorregiao",
    "UF":                           "uf",
    "Regiao do Pais":               "regiao_pais",
    "Populacao Estimada IBGE 2022": "populacao_2022",
    "Codigo Regiao de Saude":       "cod_regiao_saude",
    "Regiao de Saude":              "nome_regiao_saude",
}


def carregar_e_validar_macrorregioes(caminho: Path) -> pd.DataFrame:
    if not caminho.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")

    df = pd.read_csv(
        caminho,
        dtype={"Codigo Municipio": "string"},
        encoding="utf-8",
    )

    # Valida nomes ORIGINAIS (antes do rename)
    colunas_faltando = [k for k in COLUNAS_MACRO_RENAME if k not in df.columns]
    if colunas_faltando:
        raise ValueError(
            f"Colunas faltando:\n{colunas_faltando}\n"
            f"Encontradas:\n{df.columns.tolist()}"
        )

    # Renomeia e seleciona
    df = df.rename(columns=COLUNAS_MACRO_RENAME)
    df = df[list(COLUNAS_MACRO_RENAME.values())].copy()

    # Converte população: "2.817.381" → 2817381
    df["populacao_2022"] = (
        df["populacao_2022"]
          .astype(str)
          .str.replace(".", "", regex=False)
          .str.replace(",", "", regex=False)
          .str.strip()
          .pipe(pd.to_numeric, errors="coerce")
          .astype("Int64")
    )

    pop_nulos = df["populacao_2022"].isna().sum()
    if pop_nulos > 0:
        log.warning(f"⚠️ {pop_nulos} municípios com população nula.")

    # Valida formato 6 dígitos
    invalidos = ~df["cod_municipio"].str.match(r"^\d{6}$", na=False)
    if invalidos.any():
        exemplos = df.loc[invalidos, "cod_municipio"].head(5).tolist()
        raise ValueError(f"{invalidos.sum()} códigos inválidos: {exemplos}")

    # Valida ausência de duplicatas
    duplicatas = df["cod_municipio"].duplicated()
    if duplicatas.any():
        exemplos = df.loc[duplicatas, "cod_municipio"].head(5).tolist()
        raise ValueError(f"{duplicatas.sum()} municípios duplicados: {exemplos}")

    # Cobertura
    N_MUNICIPIOS_BR = 5570
    cobertura = len(df) / N_MUNICIPIOS_BR
    n_macros = df["cod_macrorregiao"].nunique()

    log.info(
        f"Macrorregiões OK: {len(df):,} municípios "
        f"({cobertura:.1%}) | {n_macros} macrorregiões"
    )
    if cobertura < 0.98:
        log.warning(f"⚠️ Cobertura {cobertura:.1%} — monitorar perdas no merge.")

    return df


DF_MACRO = carregar_e_validar_macrorregioes(CAMINHO_MACRO)
print(f"✅ Macrorregiões carregadas")
print(f"   Municípios: {len(DF_MACRO):,}")
print(f"   Macrorregiões únicas: {DF_MACRO['cod_macrorregiao'].nunique()}")
print(f"   Colunas finais: {list(DF_MACRO.columns)}")
print(f"\nAmostra:")
print(DF_MACRO.head(3).to_string(index=False))

17:53:46 [INFO] Macrorregiões OK: 5,571 municípios (100.0%) | 121 macrorregiões


✅ Macrorregiões carregadas
   Municípios: 5,571
   Macrorregiões únicas: 121
   Colunas finais: ['cod_municipio', 'nome_municipio', 'cod_macrorregiao', 'nome_macrorregiao', 'uf', 'regiao_pais', 'populacao_2022', 'cod_regiao_saude', 'nome_regiao_saude']

Amostra:
cod_municipio nome_municipio  cod_macrorregiao     nome_macrorregiao               uf  regiao_pais  populacao_2022  cod_regiao_saude nome_regiao_saude
       530010       Brasilia              5302      DISTRITO FEDERAL Distrito Federal Centro-Oeste         2817381             53001  DISTRITO FEDERAL
       521880      Rio Verde              5206 MACRORREGIAO SUDOESTE            Goias Centro-Oeste          225696             52015        SUDOESTE I
       521190          Jatai              5206 MACRORREGIAO SUDOESTE            Goias Centro-Oeste          105729             52016       SUDOESTE II
